In [3]:
# Example: PyTorch Geometric
from torch_geometric.datasets import QM9
dataset = QM9(root='data/QM9')

/Users/jackson/.virtualenvs/ml/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Extracting data/QM9/raw/qm9_v3.zip
Processing...
Using a pre-processed version of the dataset. Please install 'rdkit' to alternatively process the raw data.
Done!


In [ ]:
from pymatgen.core import Structure
from ase import Atoms
import networkx as nx
import itertools

# Read a CIF from COD
structure = Structure.from_file("data/CIF/1100118.cif")

# Build a simple graph
G = nx.Graph()
for i, site in enumerate(structure.sites):
    G.add_node(i, element=site.specie.symbol, coords=site.coords)

# Add edges for pairs within cutoff distance
cutoff = 3.0
for i, site in enumerate(structure):
    neighbors = structure.get_neighbors(site, r=cutoff)
    for neighbor in neighbors:
        j = neighbor.index
        if i < j:
            rel_vec = neighbor.coords - site.coords
            G.add_edge(
                i,
                j,
                weight=neighbor.nn_distance,
                rel_vec=rel_vec
            )

In [17]:
import torch
from torch_geometric.data import Data

# Node features: atomic numbers
x = torch.tensor(
    [site.specie.number for site in structure],
    dtype=torch.long
).unsqueeze(-1)

# Positions
pos = torch.tensor(
    [site.coords for site in structure],
    dtype=torch.float
)

# Edges
edge_index = []
edge_attr = []

for i, j, data in G.edges(data=True):
    edge_index.append([i, j])
    edge_index.append([j, i])  # undirected graph

    edge_attr.append(data["rel_vec"])
    edge_attr.append(-data["rel_vec"])

edge_index = torch.tensor(edge_index).t().contiguous()
edge_attr = torch.tensor(edge_attr, dtype=torch.float)

data = Data(x=x, pos=pos, edge_index=edge_index, edge_attr=edge_attr)


/var/folders/0h/6bw221td3fd034qnc35bbqhc0000gn/T/ipykernel_23782/3937064649.py:11: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:256.)
  pos = torch.tensor(


In [ ]:
import torch
import torch.nn as nn
from torch_geometric.utils import scatter

class MinimalEGNNLayer(nn.Module):
    def __init__(self, in_features, hidden_dim):
        super().__init__()
        
        self.edge_mlp = nn.Sequential(
            nn.Linear(2 * in_features + 1, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU()
        )
        
        self.node_mlp = nn.Sequential(
            nn.Linear(in_features + hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, in_features)
        )
        
        self.coord_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, h, x, edge_index):
        row, col = edge_index

        # Relative positions
        rel = x[row] - x[col]
        dist2 = (rel ** 2).sum(dim=1, keepdim=True)

        # Edge features
        edge_input = torch.cat([h[row], h[col], dist2], dim=1)
        m_ij = self.edge_mlp(edge_input)

        # Node update
        m_i = scatter(m_ij, row, dim=0, dim_size=h.size(0), reduce='add')
        h = self.node_mlp(torch.cat([h, m_i], dim=1))

        # Coordinate update (equivariant!)
        coord_update = rel * self.coord_mlp(m_ij)
        delta_x = scatter(coord_update, row, dim=0, dim_size=x.size(0), reduce='add')
        x = x + delta_x

        return h, x


ModuleNotFoundError: No module named 'torch_scatter'